# 🛠️ Feature Engineering

## Objective

The objective of this notebook is to create meaningful features that better represent engine health and operational performance.

These engineered features will improve exploratory analysis, business insights, dashboard development, and predictive maintenance modeling.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/processed/aeroengine_cleaned.csv")
df.head()

,Engine_ID,Cycle,Operational_Setting_1,Operational_Setting_2,Sensor_2,Sensor_3,Sensor_4,Sensor_6,Sensor_7,Sensor_8,Sensor_9,Sensor_11,Sensor_12,Sensor_13,Sensor_14,Sensor_15,Sensor_17,Sensor_20,Sensor_21
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044


In [3]:
engine_life = df.groupby("Engine_ID")["Cycle"].max()
df["Engine_Life"] = df["Engine_ID"].map(engine_life)
df[["Engine_ID","Cycle","Engine_Life"]].head()

,Engine_ID,Cycle,Engine_Life
0,1,1,192
1,1,2,192
2,1,3,192
3,1,4,192
4,1,5,192


### Business Insight

Engine Lifetime represents the total operational cycles completed by each engine.

This feature provides useful context when comparing engine degradation across different units.

In [4]:
df["RUL"] = df["Engine_Life"] - df["Cycle"]
df[["Engine_ID","Cycle","Engine_Life","RUL"]].head()

,Engine_ID,Cycle,Engine_Life,RUL
0,1,1,192,191
1,1,2,192,190
2,1,3,192,189
3,1,4,192,188
4,1,5,192,187


### Business Insight

Remaining Useful Life estimates how many operational cycles remain before engine failure.

This is one of the most important indicators used in predictive maintenance.

In [6]:
conditions = [
    df["RUL"] > 100,
    (df["RUL"] > 50) & (df["RUL"] <= 100),
    df["RUL"] <= 50
]

labels = [
    "Healthy",
    "Warning",
    "Critical"
]

df["Engine_Status"] = np.select(
    conditions,
    labels,
    default="Unknown"
)
df["Engine_Status"].value_counts()

Engine_Status
Healthy     10531
Critical     5100
Warning      5000
Name: count, dtype: int64

In [8]:
sensor_columns = [
    col
    for col in df.columns
    if "Sensor" in col
]
df["Average_Sensor_Value"] = df[sensor_columns].mean(axis=1)
df[
    [
        "Average_Sensor_Value"
    ]
].head()

,Average_Sensor_Value
0,1813.400567
1,1813.117693
2,1813.539467
3,1813.012140
4,1813.686920


In [9]:
df["Sensor_Variability"] = df[sensor_columns].std(axis=1)
df[
    [
        "Sensor_Variability"
    ]
].head()

,Sensor_Variability
0,2875.867171
1,2874.319802
2,2876.258108
3,2875.704060
4,2876.660958


### Business Insight

Sensor variability measures how much sensor readings differ from one another.

Higher variability may indicate unstable operating conditions.

In [10]:
df.to_csv(
    "../data/processed/aeroengine_feature_engineered.csv",
    index=False
)

| Engineered Feature   | Business Value               |
| -------------------- | ---------------------------- |
| Engine_Life          | Total engine lifespan        |
| RUL                  | Remaining operational cycles |
| Engine_Status        | Maintenance priority         |
| Average_Sensor_Value | Overall sensor behavior      |
| Sensor_Variability   | Operational stability        |
